# CNN vs Vision Transformer vs Mamba/SSM en Fashion-MNIST

Este notebook compara **tres paradigmas de visión artificial** sobre **la misma tarea**:

1. **CNN clásica**
2. **Vision Transformer (ViT) minimal**
3. **Modelo tipo Mamba / SSM minimal** *(opcional / avanzado)*

## Objetivos

- Ver que **la misma tarea** puede resolverse con arquitecturas diferentes.
- Entender qué cambia entre **convolución**, **atención** y **modelado secuencial eficiente**.
- Comparar:
  - rendimiento
  - coste conceptual
  - facilidad de explicación
  - adecuación según el problema

> Recomendación didáctica: usar primero la CNN como baseline, luego el ViT como “avance” y dejar SSM/Mamba como extensión opcional.


## 0. Idea general

### CNN
- Aprende **filtros locales**
- Comparte pesos
- Tiene **sesgos inductivos fuertes** para imágenes

### Vision Transformer (ViT)
- Divide la imagen en **patches**
- Usa **autoatención**
- Modela relaciones **globales**

### Mamba / SSM
- Trata la entrada como **secuencia**
- Sustituye parte o toda la atención por **dinámicas de estado**
- Busca **eficiencia lineal** y menor coste de memoria


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist

print("TensorFlow:", tf.__version__)


## 1. Carga y preparación del dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Para CNN y ViT: añadimos canal
x_train_img = x_train[..., None]
x_test_img = x_test[..., None]

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Train:", x_train_img.shape, y_train.shape)
print("Test :", x_test_img.shape, y_test.shape)


In [ ]:
plt.figure(figsize=(10,4))
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(class_names[y_train[i]], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()


## 2. Baseline — CNN clásica

Esta es la arquitectura más natural para empezar en visión:
- Conv2D
- MaxPooling
- Conv2D
- MaxPooling
- Flatten
- Capas densas


In [ ]:
cnn_model = models.Sequential([
    layers.Input(shape=(28,28,1)),
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()


In [ ]:
cnn_history = cnn_model.fit(
    x_train_img, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)


### Evaluación CNN

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_img, y_test, verbose=0)
print("CNN test accuracy:", cnn_test_acc)


## 2.5 Visualización de *patches* en ViT

Antes de entrenar un **Vision Transformer**, conviene ver qué significa exactamente
dividir una imagen en **patches**.

### Idea
En lugar de procesar la imagen completa con filtros convolucionales, un ViT:

1. Divide la imagen en bloques pequeños (*patches*)
2. Aplana cada parche
3. Lo convierte en un embedding
4. Trata la secuencia de patches como si fueran “tokens”

En este ejemplo usaremos **patches de 7×7**, así que una imagen de 28×28 se divide en:

- 4 patches por fila
- 4 patches por columna
- **16 patches en total**


In [ ]:
# Visualización de patches para una imagen de ejemplo
patch_size = 7

idx = 0
img = x_train_img[idx, :, :, 0]

h, w = img.shape
patches = []

for i in range(0, h, patch_size):
    for j in range(0, w, patch_size):
        patch = img[i:i+patch_size, j:j+patch_size]
        patches.append(patch)

print(f"Tamaño imagen: {img.shape}")
print(f"Tamaño patch: {patch_size}x{patch_size}")
print(f"Número total de patches: {len(patches)}")

# Imagen original con rejilla
plt.figure(figsize=(4,4))
plt.imshow(img, cmap="gray")
for i in range(0, h, patch_size):
    plt.axhline(i - 0.5, color="red", linewidth=1)
for j in range(0, w, patch_size):
    plt.axvline(j - 0.5, color="red", linewidth=1)
plt.title(f"Imagen original dividida en patches ({class_names[y_train[idx]]})")
plt.axis("off")
plt.show()

# Mostrar los patches por separado
plt.figure(figsize=(8,8))
for k, patch in enumerate(patches):
    plt.subplot(h // patch_size, w // patch_size, k + 1)
    plt.imshow(patch, cmap="gray")
    plt.title(f"P{k}", fontsize=8)
    plt.axis("off")
plt.suptitle("Patches individuales que verá el ViT", y=0.92)
plt.tight_layout()
plt.show()


### Interpretación didáctica

- Una **CNN** ve la imagen completa y aprende filtros locales que se deslizan sobre ella.
- Un **ViT** rompe la imagen en piezas fijas y trabaja con ellas como si fueran una secuencia.
- Eso permite reutilizar la idea de los Transformers de NLP:
  - token → embedding
  - aquí: patch → embedding

### Preguntas para clase

1. ¿Qué información espacial se pierde al aplanar un patch?
2. ¿Por qué hacen falta **positional embeddings**?
3. ¿Qué ventaja tiene este enfoque frente a una CNN?
4. ¿Qué desventaja puede tener cuando hay pocos datos?


## 3. Vision Transformer (ViT) minimal

Aquí tratamos la imagen como una **secuencia de patches**.

### Pipeline
1. Dividir imagen en patches
2. Aplanar patches
3. Proyectarlos a embeddings
4. Añadir embeddings posicionales
5. Pasarlos por bloques Transformer
6. Clasificar


In [ ]:
patch_size = 7
num_patches = (28 // patch_size) ** 2   # 16 patches
projection_dim = 64
num_heads = 4
transformer_layers = 2

class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

def create_vit_classifier():
    inputs = layers.Input(shape=(28, 28, 1))
    patches = Patches(patch_size)(inputs)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        x2 = layers.Add()([attention_output, encoded_patches])

        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=[projection_dim * 2, projection_dim], dropout_rate=0.1)
        encoded_patches = layers.Add()([x3, x2])

    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.2)(representation)
    features = mlp(representation, hidden_units=[128], dropout_rate=0.2)
    logits = layers.Dense(10, activation="softmax")(features)

    return tf.keras.Model(inputs=inputs, outputs=logits)

vit_model = create_vit_classifier()
vit_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

vit_model.summary()


In [ ]:
vit_history = vit_model.fit(
    x_train_img, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)


### Evaluación ViT

In [ ]:
vit_test_loss, vit_test_acc = vit_model.evaluate(x_test_img, y_test, verbose=0)
print("ViT test accuracy:", vit_test_acc)


## 4. Modelo tipo Mamba / SSM minimal (opcional)

Esto es un **avance** de arquitecturas modernas que intentan evitar el coste cuadrático de la atención.

### Idea
- Reordenar la imagen como secuencia
- Procesarla con una capa tipo **State Space Model**
- Mantener un coste más eficiente

> Nota: la disponibilidad depende de si tienes `keras_nlp` y su capa SSM en tu entorno.


In [ ]:
# Intentamos usar keras_nlp si está disponible
try:
    import keras_nlp
    HAS_KERAS_NLP = True
except Exception:
    HAS_KERAS_NLP = False

print("keras_nlp disponible:", HAS_KERAS_NLP)


In [ ]:
if HAS_KERAS_NLP and hasattr(keras_nlp.layers, "StateSpaceModel"):
    ssm_inputs = layers.Input(shape=(28,28,1))
    x = layers.Reshape((784, 1))(ssm_inputs)

    x = keras_nlp.layers.StateSpaceModel(
        units=64,
        kernel_size=16,
        activation="gelu"
    )(x)

    x = layers.LayerNormalization()(x)
    x = layers.Flatten()(x)
    ssm_outputs = layers.Dense(10, activation="softmax")(x)

    ssm_model = tf.keras.Model(ssm_inputs, ssm_outputs)
    ssm_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    ssm_model.summary()
else:
    print("No se puede construir el modelo SSM en este entorno.")


In [ ]:
if HAS_KERAS_NLP and hasattr(keras_nlp.layers, "StateSpaceModel"):
    ssm_history = ssm_model.fit(
        x_train_img, y_train,
        validation_split=0.1,
        epochs=5,
        batch_size=128,
        verbose=1
    )

    ssm_test_loss, ssm_test_acc = ssm_model.evaluate(x_test_img, y_test, verbose=0)
    print("SSM/Mamba-like test accuracy:", ssm_test_acc)
else:
    print("Se omite entrenamiento SSM en este entorno.")


## 5. Comparativa de curvas de entrenamiento

Si algún modelo no se ha podido entrenar, simplemente compara CNN y ViT.


In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(cnn_history.history["val_accuracy"], label="CNN")
plt.plot(vit_history.history["val_accuracy"], label="ViT")
if "ssm_history" in globals():
    plt.plot(ssm_history.history["val_accuracy"], label="SSM/Mamba-like")
plt.title("Validation Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(cnn_history.history["val_loss"], label="CNN")
plt.plot(vit_history.history["val_loss"], label="ViT")
if "ssm_history" in globals():
    plt.plot(ssm_history.history["val_loss"], label="SSM/Mamba-like")
plt.title("Validation Loss")
plt.legend()

plt.tight_layout()
plt.show()


## 6. Tabla de resultados

Interpretación rápida:
- **CNN**: muy buena baseline, eficiente y fácil de explicar.
- **ViT**: más flexible, más “global”, pero suele necesitar más datos para brillar.
- **SSM/Mamba-like**: muy prometedor en eficiencia, todavía menos estándar en docencia inicial.


In [ ]:
results = {
    "CNN": cnn_test_acc,
    "ViT": vit_test_acc,
}

if "ssm_test_acc" in globals():
    results["SSM/Mamba-like"] = ssm_test_acc

for name, acc in results.items():
    print(f"{name:20s} -> test_acc = {acc:.4f}")


## 7. Conclusiones didácticas

### ¿Qué enseñar primero?
- **Primero CNN**
- Luego **ViT como avance**
- Y **SSM/Mamba** como extensión opcional

### Mensaje clave para el alumnado
No hay una única arquitectura “ganadora” universal:

- **CNN**: fuertes sesgos inductivos, buena con pocos datos
- **ViT**: menos sesgo inductivo, mejor con datos masivos
- **SSM/Mamba**: busca eficiencia y memoria lineal

### Preguntas para clase
1. ¿Por qué la CNN suele ser más fácil de entrenar aquí?
2. ¿Qué gana un ViT al trabajar por patches?
3. ¿Qué problema intenta resolver un modelo tipo Mamba?
4. ¿Qué arquitectura usarías en un móvil? ¿Y en un gran cluster?
